In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss
from catboost import CatBoostClassifier


TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"
SUB_PATH = "../data/sample_submission.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train["Target"] = (train["Heart Disease"] == "Presence") * 1
train = train.drop(["id", "Heart Disease"], axis=1)
test = test.drop(["id"], axis=1)

TARGET_COL = train.columns[-1]
FEATURE_COLS = list(train.columns)[:-1]
X = train[FEATURE_COLS].copy()
y = train[TARGET_COL].copy()
X_test = test[FEATURE_COLS].copy()

CAT_COLS = ["Sex", "Chest pain type", "FBS over 120", "EKG results", "Exercise angina", "Slope of ST", "Number of vessels fluro", "Thallium"]
for c in CAT_COLS:
    X[c] = X[c].astype("string")
    X_test[c] = X_test[c].astype("string")
cat_features = [X.columns.get_loc(c) for c in CAT_COLS]


N_SPLITS = 5
RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

test_pred = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=3000,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5.0,
        random_strength=1.0,
        bagging_temperature=1.0,
        border_count=128,
        random_seed=RANDOM_STATE + fold,
        verbose=200,
        task_type="GPU",
        devices="0",
        auto_class_weights="Balanced",
    )

    model.fit(
        X_train, y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        early_stopping_rounds=200
    )

    test_pred += model.predict_proba(X_test)[:, 1] / N_SPLITS

sub = pd.read_csv(SUB_PATH)
